Import Modules

In [2]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from itertools import combinations
import json

import helpers

Data cleaning

In [3]:
#Import Data
data = pd.read_csv("iupac_high-confidence_v2_3.csv")

#Drop irrelivant columns
data = data.drop(axis=1, labels=['ref','ref_remarks','remarks', 'method', 'entry_remarks', 'unique_ID', 'num_name_contributors', 
                                 'original_IUPAC_nicknames', 'name_contributors','source', 'cosolvent', 'original_T', 'acidity_label']) #Acidity_label is dropped because it's consistent with pka_type

#Drop NaN entries
data['T'] = pd.to_numeric(data['T'], errors='coerce')
data['pka_value'] = pd.to_numeric(data['pka_value'], errors='coerce')
data = data.dropna(subset=['pka_value', 'pka_type', 'T', 'assessment'])

#The pressure column reports datapoints measured at high pressure, significantly changing the pKa value. NaN values likely to indicate atmospheric pressure.
#The pressure values at 1 bar is for acetic acid at different temperatures, which can also be dropped
data = data[data['pressure'].isna()]
data = data.drop(axis=1, labels = ['pressure'])
#The pKa value doesn't change significantly in this range
data = data[data['T'].apply(lambda x: 20 < x and x < 30)]
data = data.drop(axis=1, labels = ['T'])

#Only data with reliable or approximate assessment is kept. They are ranked based on their assessment, and duplicate entries for a the same structure and pka_type are removed, only keeping the first.
data = data[data['assessment'].isin(['Reliable', 'Approximate', 'Uncertain'])]
order = {'Reliable':1, "Approximate":2, 'Uncertain':3}
data['assessment'] = data['assessment'].map(order)

#Entries with similar SMILES, InChI, pka_type, and highest assessment are averaged. The first original_IUPAC_names of each duplicate entry is chosen.
best_assessment = data.groupby(['SMILES', 'InChI', 'pka_type'])['assessment'].min().reset_index()
data = data.merge(best_assessment, on=['SMILES', 'InChI', 'assessment', 'pka_type'], how='inner')
data = data.groupby(['SMILES', 'InChI', 'assessment', 'pka_type'], as_index=False).agg({
    'pka_value': 'mean',
    'original_IUPAC_names': 'first'
})

#Only structures with 2 acid/base sites or less are kept. All structures with more acid/base sites or ambiguous labeling are dropped. pKb is also dropped because it's a minority
ab_count = data.groupby(['SMILES'])['pka_type'].nunique()
valid_SMILES = ab_count[ab_count < 3].index
data = data[data['SMILES'].isin(valid_SMILES)]

#Di-acids and Di-bases are kept but with only their strongest acid/base site. So their pKa2 and pKaH2 are dropped.
data = data[data['pka_type'].isin(['pKa1', 'pKaH1'])]

#Check valid smiles
data = data[data['SMILES'].apply(helpers.check_smiles)]


Add identification for smiles

In [4]:
#Count the number of carbons in structure
data['num_carbons'] = data['SMILES'].apply(helpers.count_carbons)

#Count the number of heavy atoms (all atoms except hydrogen) in structure
data['num_heavy_atoms'] = data['SMILES'].apply(helpers.count_heavy_atoms)
#Drop structures that are too large
data = data[data['num_heavy_atoms'] < 25]
data = data.drop(axis=1, columns=['num_heavy_atoms'])

#Check if structure is amphoteric (is both an acid and a base - having both a pka and a pkah value)
data = data.groupby('SMILES', group_keys=False).apply(helpers.check_amphoteric)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13884\884873287.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('SMILES', group_keys=False).apply(helpers.check_amphoteric)


In [5]:
#Separate into sub dataframes
df_amphoterics = data[data['amphoteric']]
data = data[~data['amphoteric']]
df_acids = data[data['pka_type'].isin(['pKa1'])].copy()
df_bases = data[data['pka_type'].isin(['pKaH1'])].copy()

df_acids = df_acids.drop(columns=['amphoteric'])
df_bases = df_bases.drop(columns=['amphoteric'])
df_amphoterics = df_amphoterics.drop(columns=['amphoteric'])

#All amphoterics should have a nitrogen to act as a base. If not, it is removed from amphoterics and added back to the acid df
acids_wrong_amp = df_amphoterics[df_amphoterics['SMILES'].apply(lambda x: 'N' not in x and 'n' not in x)]
df_amphoterics = df_amphoterics[df_amphoterics['SMILES'].apply(lambda x: 'N' in x or 'n' in x)]
acids_wrong_amp =  acids_wrong_amp[acids_wrong_amp['pka_type'] == 'pKa1']
df_acids = pd.concat([df_acids, acids_wrong_amp], ignore_index=True)

#Only amphoterics with at least a difference between pka and pkah larger than 3 are kept to avoid ambiguity
amphoterics_pka_diff = df_amphoterics.groupby('SMILES')['pka_value'].agg(lambda x: abs(x.max() - x.min()))
valid_amphoterics = amphoterics_pka_diff[amphoterics_pka_diff >= 3].index
df_amphoterics = df_amphoterics[df_amphoterics['SMILES'].isin(valid_amphoterics)]

#All bases should not have the carboxylic group
df_bases = df_bases[df_bases['SMILES'].apply(helpers.check_CO2H_base)]

In [6]:
df_acids['priority_functional_group'] = df_acids['SMILES'].apply(helpers.detect_functional_groups_acid)
df_bases['priority_functional_group'] = df_bases['SMILES'].apply(helpers.detect_functional_groups_base)
df_amphoterics['priority_functional_group'] = df_amphoterics['SMILES'].apply(helpers.detect_functional_groups_acid)

#Any structure that have a non-defined functional group is dropped
df_acids = df_acids.dropna(subset='priority_functional_group')
df_bases = df_bases.dropna(subset='priority_functional_group')
df_amphoterics = df_amphoterics.dropna(subset='priority_functional_group')

#pka and pkah of the amphoterics are separated into their own df
df_amp_pka = df_amphoterics[df_amphoterics['pka_type'].isin(['pKa1'])].copy()
df_amp_pkah = df_amphoterics[df_amphoterics['pka_type'].isin(['pKaH1'])].copy()

#These are the only two major functional groups in the amphoterics
df_amp_pka = df_amp_pka[df_amp_pka['priority_functional_group'].isin(['CO2H', 'NH'])]
df_amp_pkah = df_amp_pkah[df_amp_pkah['priority_functional_group'].isin(['CO2H', 'NH'])]

In [7]:
testtao = df_acids[df_acids['priority_functional_group'] == '1,3-DICARB']
lom = Chem.MolFromSmarts('[n][OH]')
testtao = df_acids[df_acids['SMILES'].apply(lambda x: Chem.MolFromSmiles(x).HasSubstructMatch(lom))]
testtao

,SMILES,InChI,assessment,pka_type,pka_value,original_IUPAC_names,num_carbons,priority_functional_group
1679,On1ccccc1=S,"InChI=1S/C5H5NOS/c7-6-4-2-1-3-5(6)8/h1-4,7H",3,pKa1,4.6,"pyridine-2(1H)-thione, 1-hydroxy-",5,N-OH


Generate valid pairs

In [8]:
#Group into dictionaries
acids_dict = {name: group for name, group in df_acids.groupby('priority_functional_group')}
bases_dict = {name: group for name, group in df_bases.groupby('priority_functional_group')}
amp_pka_dict = {name: group for name, group in df_amp_pka.groupby('priority_functional_group')}
amp_pkah_dict = {name: group for name, group in df_amp_pkah.groupby('priority_functional_group')}

In [9]:
#Generate valid pairs
acid_pairs = helpers.generate_ordered_pairs(acids_dict)
base_pairs = helpers.generate_ordered_pairs(bases_dict)
amp_pka_pairs = helpers.generate_ordered_pairs(amp_pka_dict)
amp_pkah_pairs = helpers.generate_ordered_pairs(amp_pkah_dict)

In [11]:
all_pairs_data = {'acids': acid_pairs, 
                  'bases': base_pairs, 
                  'amphoterics_pka': amp_pka_pairs, 
                  'amphoterics_pkah': amp_pkah_pairs}

with open('all_pairs_data.json', 'w') as f:
    json.dump(all_pairs_data, f, indent=4)